# Data Cleaner 
#### All functions related to Data Cleaning are researched, developed and tested here
#### finally past in data_cleaner.py file for scalability and faster performance in the backend.

In [1]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor,KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor,DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")
import seaborn  as sns
from sklearn.datasets import *

In [2]:
'''df = pd.DataFrame({
    'A': [1, 2, None, 4, 5],
    'B': [None, 3, 4, 5, 6],
    'C': [2, None, 4, 5, None],
    'D': ["Hi",None,"H ello","How","Are"],
    'E': ["2025-11-25","2005-10-24","2024-9-24","2005-10-24",None]
})'''
df=pd.read_csv("Titanic-Dataset.csv")
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [3]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

## Ai validation for the datatypes of columns
For most of the columns it can give accurate results.
 if not then dtypes are asked in the Ui for confirmation and will make modifications according to it.

In [4]:
import requests

def call_llm(prompt):
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "gemma3:4b",
                "prompt": prompt,
                "stream": False
            },
            timeout=300
        )
        
        # Check HTTP status
        if response.status_code != 200:
            print(f"Error: HTTP {response.status_code}")
            print(f"Response: {response.text}")
            return None
            
        response_json = response.json()
        
        # Check if response has the expected key
        if "response" in response_json:
            return response_json["response"]
        else:
            print(f"Error: 'response' key not found in API response")
            print(f"Available keys: {response_json.keys()}")
            print(f"Full response: {response_json}")
            return None
            
    except requests.exceptions.ConnectionError:
        print("Error: Cannot connect to Ollama at http://localhost:11434")
        print("Make sure Ollama is running: ollama serve")
        return None
    except Exception as e:
        print(f"Error: {type(e).__name__}: {e}")
        return None


In [5]:
def take_llm_decision(df):
    import json
    import re
    samples_list = {}
    column_name_list = []
    colum_before_dtype=[]
    for col in df.columns:
        column_name_list.append(col)
        samples = (df[col].astype(str).sample(5).tolist())
        samples_list[col] = samples
        colum_before_dtype.append(str(df[col].dtype))

    prompt_prefix = (
        f"Column names: {column_name_list}\n"
        f"Sample values from each column (JSON): {json.dumps(samples_list)}\n"
        f"Previous  dtype for the column: {json.dumps(colum_before_dtype)}\n"
    )

    prompt_body = (
        "You are a senior data analyst with 10 years of experience.\n"
        "Based on column names, sample values,previous data type for the column infer the most appropriate datatype.\n"
        "Do not assume the data is cleaned.\n"
        "Dates may appear as strings but should be classified as datetime.\n\n"
        "Return ONLY a valid JSON array.\n"
        "Each object must contain:\n"
        "- columnName (string)\n"
        "- datatype (one of: int, float, string, boolean, datetime, category)\n"
        "- confidence_level (number between 0 and 1)\n"
    )
    prompt_example = ('[{"columnName": "order_date", "datatype": "datetime", "confidence_level": 0.95}]')
    prompt_ = (prompt_prefix+ prompt_body+ "Example output format:\n"+ prompt_example)
    result = call_llm(prompt_)

    try:
        match = re.search(r"\[.*\]", result, re.S)
        if not match:
            return None

        data = json.loads(match.group())
        return data

    except Exception:
        return None


In [6]:
# AI Generated

import json
import re

result = take_llm_decision(df)

if isinstance(result, str):

    sections = result.split('columnName:')[1:]  
    data = []
    
    for section in sections:
        obj = {"columnName": ""}
        lines = section.strip().split('\n')
        
        for line in lines:
            if line.startswith('datatype:'):
                obj['datatype'] = line.replace('datatype:', '').strip()
            elif line.startswith('confidence level:'):
                try:
                    obj['confidence_level'] = float(line.replace('confidence level:', '').strip())
                except:
                    obj['confidence_level'] = line.replace('confidence level:', '').strip()
            elif obj['columnName'] == '':
                obj['columnName'] = line.strip().split()[0] 
        data.append(obj)
else:
    data = result

#print(json.dumps(data, indent=2))

In [7]:
datatype = data[0]['datatype']
confidence = data[1]['confidence_level'] 

for col in data:
    print(f'''Column {col['columnName']}: {col['datatype']} (confidence: {col['confidence_level']})''')

Column PassengerId: string (confidence: 0.99)
Column Survived: int (confidence: 0.99)
Column Pclass: int (confidence: 0.99)
Column Name: string (confidence: 0.99)
Column Sex: string (confidence: 0.99)
Column Age: float (confidence: 0.99)
Column SibSp: int (confidence: 0.99)
Column Parch: int (confidence: 0.99)
Column Ticket: string (confidence: 0.99)
Column Fare: float (confidence: 0.99)
Column Cabin: string (confidence: 0.99)
Column Embarked: string (confidence: 0.99)


In [8]:
colums_dtype_list={}
for col in data:
    colums_dtype_list[col['columnName']]=col['datatype']
colums_dtype_list

{'PassengerId': 'string',
 'Survived': 'int',
 'Pclass': 'int',
 'Name': 'string',
 'Sex': 'string',
 'Age': 'float',
 'SibSp': 'int',
 'Parch': 'int',
 'Ticket': 'string',
 'Fare': 'float',
 'Cabin': 'string',
 'Embarked': 'string'}

In [9]:
def type_casting(df, colums_dtype_list):
    for col in colums_dtype_list:
        if colums_dtype_list[col] in ("float", "integer", "int", "float64", "int64"):
            df[col] = pd.to_numeric(df[col], errors='coerce')
        elif colums_dtype_list[col] == "datetime":
            df[col]=df[col].astype(str).str.replace("-","/")
            df[col]=pd.to_datetime(df[col],dayfirst=True,format='mixed')
        elif colums_dtype_list[col] == 'string':
            df[col] = df[col].astype('object').astype(str)
    return df

In [10]:
type_casting(df,colums_dtype_list)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [11]:
df.dtypes

PassengerId        str
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

## Data Cleaning 
### After getting the dataset with correct dtypes here starts the data cleaning with standard methods

In [12]:
categorical_columns=[]
numerical_columns=[]
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numerical_columns.append(col)
    else:
        categorical_columns.append(col)

In [13]:
def fill_null_values_KNN(df,numerical_columns,categorical_columns):
    
    imputer = IterativeImputer(estimator=RandomForestRegressor(),max_iter=10,random_state=42)
    df[numerical_columns] = imputer.fit_transform(df[numerical_columns])
    imputer = SimpleImputer(strategy='most_frequent')
    df[categorical_columns] = imputer.fit_transform(df[categorical_columns])
    for col in categorical_columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            s = pd.to_datetime(df[col], errors='coerce')
            ints = s.map(lambda x: x.value if pd.notnull(x) else np.nan).astype('float')
            ints = pd.Series(ints, index=df.index)
            ints_interp = ints.interpolate(method='linear', limit_direction='both')
            result = pd.Series([pd.NaT] * len(df), index=df.index, dtype='datetime64[ns]')
            mask = ints_interp.notna()
            if mask.any():
                result.loc[mask] = pd.to_datetime(ints_interp[mask].astype('int64'), unit='ns')
            df[col] = result
    return df

In [14]:
def standardizevalues(df,categorical_columns):
    for col in categorical_columns:
        if  not df[col].dtype=='datetime64[ns]':
            df[col] = df[col].str.strip()
            df[col] = df[col].str.replace(r'\s+', '', regex=True)
    return df

In [15]:
def remove_duplicates(df):
    df=df.drop_duplicates()
    return df

In [16]:
df = fill_null_values_KNN(df,numerical_columns,categorical_columns)
df = standardizevalues(df,categorical_columns)
df=remove_duplicates(df)
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0.0,3.0,"Braund,Mr.OwenHarris",male,22.000000,1.0,0.0,A/521171,7.2500,B96B98,S
1,2,1.0,1.0,"Cumings,Mrs.JohnBradley(FlorenceBriggsThayer)",female,38.000000,1.0,0.0,PC17599,71.2833,C85,C
2,3,1.0,3.0,"Heikkinen,Miss.Laina",female,26.000000,0.0,0.0,STON/O2.3101282,7.9250,B96B98,S
3,4,1.0,1.0,"Futrelle,Mrs.JacquesHeath(LilyMayPeel)",female,35.000000,1.0,0.0,113803,53.1000,C123,S
4,5,0.0,3.0,"Allen,Mr.WilliamHenry",male,35.000000,0.0,0.0,373450,8.0500,B96B98,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0.0,2.0,"Montvila,Rev.Juozas",male,27.000000,0.0,0.0,211536,13.0000,B96B98,S
887,888,1.0,1.0,"Graham,Miss.MargaretEdith",female,19.000000,0.0,0.0,112053,30.0000,B42,S
888,889,0.0,3.0,"Johnston,Miss.CatherineHelen""Carrie""",female,24.381667,1.0,2.0,W./C.6607,23.4500,B96B98,S
889,890,1.0,1.0,"Behr,Mr.KarlHowell",male,26.000000,0.0,0.0,111369,30.0000,C148,C


In [17]:
type_casting(df,colums_dtype_list)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0.0,3.0,"Braund,Mr.OwenHarris",male,22.000000,1.0,0.0,A/521171,7.2500,B96B98,S
1,2,1.0,1.0,"Cumings,Mrs.JohnBradley(FlorenceBriggsThayer)",female,38.000000,1.0,0.0,PC17599,71.2833,C85,C
2,3,1.0,3.0,"Heikkinen,Miss.Laina",female,26.000000,0.0,0.0,STON/O2.3101282,7.9250,B96B98,S
3,4,1.0,1.0,"Futrelle,Mrs.JacquesHeath(LilyMayPeel)",female,35.000000,1.0,0.0,113803,53.1000,C123,S
4,5,0.0,3.0,"Allen,Mr.WilliamHenry",male,35.000000,0.0,0.0,373450,8.0500,B96B98,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0.0,2.0,"Montvila,Rev.Juozas",male,27.000000,0.0,0.0,211536,13.0000,B96B98,S
887,888,1.0,1.0,"Graham,Miss.MargaretEdith",female,19.000000,0.0,0.0,112053,30.0000,B42,S
888,889,0.0,3.0,"Johnston,Miss.CatherineHelen""Carrie""",female,24.381667,1.0,2.0,W./C.6607,23.4500,B96B98,S
889,890,1.0,1.0,"Behr,Mr.KarlHowell",male,26.000000,0.0,0.0,111369,30.0000,C148,C


In [18]:
df.dtypes

PassengerId        str
Survived       float64
Pclass         float64
Name               str
Sex                str
Age            float64
SibSp          float64
Parch          float64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

## Creating a cleaned dataset file to send for EDA.

In [19]:
df.to_csv("cleaned_titanic.csv")